# 🏆 [Day 33 과제 LV3(통합)] 엔터프라이즈 GDS 복합 투영 및 다차원 중심성 분석

> **미션 개요**:
> Hetionet 바이오 메디컬 지식그래프와 서울 지하철 환승 네트워크를 활용하여 **실전 GDS 인메모리 투영, 다중 관계 가중치 계산, PageRank/개인화 PageRank/매개 중심성 비교 분석**을 완벽히 마스터합니다.
>
> 1. **다중 이종 노드·관계 복합 투영**: 질병-유전자-약물(Disease-Gene-Compound) 삼각 결합망 네이티브 투영
> 2. **다차원 중심성 3종 교차 비교**: 차수(Degree) vs PageRank vs 매개 중심성(Betweenness) 순위 상관관계 분석
> 3. **타겟 질환 중심의 개인화 PageRank (PPR)**: 특정 암종(예: 유방암, 폐암) 관점에서의 유망 타겟 유전자 및 약물 후보군 우선순위 도출
> 4. **GDS 인메모리 수명주기(Lifecycle) 거버넌스**: 메모리 추정(Estimate), 목록 조회(List), 안전한 메모리 해제(Drop) 100% 보장

## 0. 환경 설정 및 Neo4j GDS 연결

In [ ]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv(".env")
load_dotenv("../.env")

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def run_cypher(query, **params):
    with driver.session() as session:
        return [record.data() for record in session.run(query, **params)]

# GDS 버전 확인
ver_info = run_cypher("RETURN gds.version() AS ver")
print(f"✅ Neo4j GDS 플러그인 버전: {ver_info[0]['ver']}")

## 1. 다중 이종 노드·관계 복합 인메모리 투영 (gds.graph.project)
- 질병(`Disease`), 유전자(`Gene`), 화합물(`Compound`) 3개 노드 레이블
- 질병-유전자 연관(`ASSOCIATES_DaG`) 및 화합물-유전자 결합(`BINDS_CbG`)을 모두 `UNDIRECTED`로 투영합니다.

In [ ]:
# 기존 투영 정리
for g in run_cypher("CALL gds.graph.list() YIELD graphName RETURN graphName"):
    run_cypher(f"CALL gds.graph.drop('{g['graphName']}') YIELD graphName")

proj_cypher = """
CALL gds.graph.project(
    'tripletGraph',
    ['Disease', 'Gene', 'Compound'],
    {
        DaG: {type: 'ASSOCIATES_DaG', orientation: 'UNDIRECTED'},
        CbG: {type: 'BINDS_CbG', orientation: 'UNDIRECTED'}
    }
)
YIELD graphName, nodeCount, relationshipCount, projectMillis
"""
res = run_cypher(proj_cypher)[0]
print(f"✅ 투영 완료: {res['graphName']} (노드: {res['nodeCount']:,}개, 엣지: {res['relationshipCount']:,}건)")

# 검산 assert
assert res['nodeCount'] > 0, "노드 투영 실패"
assert res['relationshipCount'] > 0, "관계 투영 실패"

## 2. 3대 중심성 지표(Degree, PageRank, Betweenness) 교차 비교 분석

In [ ]:
# 1. Degree Centrality
deg_data = run_cypher("""
CALL gds.degree.stream('tripletGraph')
YIELD nodeId, score AS degree
WITH gds.util.asNode(nodeId) AS n, degree
RETURN n.name AS name, labels(n)[0] AS type, toInteger(degree) AS degree
ORDER BY degree DESC LIMIT 10
""")

# 2. PageRank Centrality
pr_data = run_cypher("""
CALL gds.pageRank.stream('tripletGraph', {maxIterations: 20, dampingFactor: 0.85})
YIELD nodeId, score AS pr_score
WITH gds.util.asNode(nodeId) AS n, pr_score
RETURN n.name AS name, labels(n)[0] AS type, round(pr_score, 4) AS pr_score
ORDER BY pr_score DESC LIMIT 10
""")

print("📊 [차수 중심성 Top 10]")
display(pd.DataFrame(deg_data))

print("\n🌐 [PageRank Top 10]")
display(pd.DataFrame(pr_data))

## 3. 특정 질병 관점의 개인화 PageRank (PPR) 신약 후보군 탐색
- 특정 질환(예: `'breast cancer'`)을 `sourceNodes`로 설정하여 가장 강하게 연결되는 유전자 및 약물 랭킹을 추출합니다.

In [ ]:
ppr_data = run_cypher("""
MATCH (d:Disease {name: 'breast cancer'})
WITH collect(id(d)) AS sources
CALL gds.pageRank.stream('tripletGraph', {
    maxIterations: 20,
    dampingFactor: 0.85,
    sourceNodes: sources
})
YIELD nodeId, score
WITH gds.util.asNode(nodeId) AS n, score
WHERE n:Compound
RETURN n.name AS compound_name, round(score, 6) AS ppr_score
ORDER BY ppr_score DESC
LIMIT 10
""")

print("🎯 [유방암(breast cancer) 타겟 개인화 PageRank Top 10 약물 후보]")
df_ppr = pd.DataFrame(ppr_data)
display(df_ppr)
assert len(df_ppr) == 10, "개인화 PageRank 결과 누락"

## 4. 인메모리 투영 메모리 안전 해제 (Drop)

In [ ]:
drop_res = run_cypher("CALL gds.graph.drop('tripletGraph') YIELD graphName")[0]
print(f"✅ 인메모리 투영 정상 해제 완료: {drop_res['graphName']}")